# 1. Models and contracts

**Learning objective:** understand the validated data contracts that every other
part of Mosaic Pathway depends on, and why they are defined before any model call
or retrieval code exists.

**Where this fits:** the whole system is a pipeline.

```
family intake -> retrieval query -> retrieved records -> generation -> grounded result
```

Every arrow in that pipeline is a contract. `mosaic_pathway.models` holds all of
them in one module, so the retriever, the generator, the evaluation suite, the
Streamlit app, and the FastAPI service all speak the same language.

## Why contracts come before the interesting parts

A retrieval-augmented generation system has three places where things silently go
wrong: the data coming in, the evidence coming back, and the text going out.

Pydantic models turn each of those into a loud, early failure:

* an intake with a missing age fails at the form boundary, not inside a prompt
* a source record with empty text fails at index build time, not at query time
* a pathway with one resource instead of two fails before a family ever sees it

The rule this project follows is **validate at boundaries**. Inside the pipeline
we pass model instances around and trust them. At the edges (form input, file
extraction, model output, HTTP request bodies) we validate.

In [ ]:
from pydantic import ValidationError

from mosaic_pathway.models import (
    ChildProfile,
    CommunitySuggestion,
    FamilyIntake,
    GroundedPathwayResult,
    LearningPathway,
    ResourceRecommendation,
    RetrievedRecord,
    RhythmPractice,
    SourceRecord,
)
from mosaic_pathway.retrieval import inventory_source_id

print("models imported")

## `ChildProfile`: the smallest contract

One child, with an age the domain actually allows and at least one interest.
The interests are not decoration: they are what makes a pathway feel personal,
and the evaluation suite later checks that they show up in the generated text.

In [ ]:
older_child = ChildProfile(
    label="older child",
    age=12,
    interests=["animals", "drawing"],
    learning_needs=["movement breaks"],
)

print(older_child.label, older_child.age)
print(older_child.interests)
print(older_child.learning_needs)

## `FamilyIntake`: what the family actually tells us

The intake is deliberately shaped around the questions a family can answer:
what they are leaving behind, what they want to keep, what they want to add,
what they value, and what constrains them.

Four of those lists are required with at least one entry. That is a product
decision encoded as a contract: a pathway built from nothing is not personal.

In [ ]:
one_child_intake = FamilyIntake(
    children=[older_child],
    leaving_behind=["rigid daily schedules"],
    wants_to_preserve=["reading together after dinner"],
    wants_to_add=["more time outdoors"],
    family_values=["curiosity", "gentleness"],
    practical_constraints=["one working parent at home"],
    additional_context="We are in our first year of self-directed learning.",
)

print("children:", len(one_child_intake.children))
print("values:", one_child_intake.family_values)
print("constraints:", one_child_intake.practical_constraints)

In [ ]:
younger_child = ChildProfile(
    label="younger child",
    age=7,
    interests=["building", "dinosaurs"],
)

two_child_intake = FamilyIntake.model_validate(
    one_child_intake.model_dump()
    | {"children": [older_child.model_dump(), younger_child.model_dump()]}
)

for child in two_child_intake.children:
    print(f"{child.label} (age {child.age}): {', '.join(child.interests)}")

## Serialization is part of the contract

The same model is used as a prompt payload, an HTTP request body, and a saved
evaluation case. Because the contract is one Pydantic model, all three of those
are the same JSON shape, and a round trip is lossless.

In [ ]:
payload = one_child_intake.model_dump_json(indent=2)

print(payload[:300])
print("...")
print(
    "round trip preserves the intake:",
    FamilyIntake.model_validate_json(payload) == one_child_intake,
)

## Failure path: an age the domain does not accept

`ChildProfile.age` is constrained to 2 through 21. A toddler or an adult is out
of scope for this MVP, so the contract refuses it rather than quietly producing
an inappropriate pathway.

In [ ]:
try:
    ChildProfile(label="baby", age=1, interests=["music"])
except ValidationError as error:
    for item in error.errors():
        print(list(item["loc"]), "->", item["msg"])

## `SourceRecord`: evidence, not documents

A `SourceRecord` is not a whole document. It is one retrievable chunk with
enough metadata to explain where it came from and who is speaking.

Two identities matter, and confusing them causes real bugs later:

* the **chunk identity** `mosaic-guide-0007` is what a pathway cites
* the **source identity** `mosaic-guide` is what the retriever uses to stop one
  document from dominating the results

In [ ]:
record = SourceRecord(
    source_id="synthetic-guide-0007",
    title="Synthetic guide, chunk 7",
    source_file="synthetic-guide.docx",
    content_type="practical_guidance",
    authority_type="mosaic_guidance",
    topics=["rhythm", "outdoor learning"],
    age_min=6,
    age_max=14,
    text="A short synthetic passage describing a gentle weekly rhythm for a family.",
)

print("chunk identity :", record.source_id)
print("source identity:", inventory_source_id(record))
print("age range      :", record.age_min, "to", record.age_max)

`age_min` and `age_max` are optional, but a model validator rejects an inverted
range. This is the pattern used throughout the project: field constraints for
simple rules, a model validator for rules that involve more than one field.

In [ ]:
try:
    SourceRecord(
        source_id="synthetic-guide-0008",
        title="Synthetic guide, chunk 8",
        source_file="synthetic-guide.docx",
        content_type="resource",
        authority_type="external_resource",
        topics=["reading"],
        age_min=14,
        age_max=6,
        text="A synthetic passage with an impossible age range attached to it.",
    )
except ValidationError as error:
    print(error.errors()[0]["msg"])

## `LearningPathway`: the shape of the answer

This is the most opinionated contract in the project. It is also the schema the
language model is asked to fill in, so its constraints are simultaneously a
product decision and a generation guardrail:

* 2 to 6 rhythm practices, so the pathway is a start and not a curriculum
* 2 to 3 resources, each with a `source_id` that must be traceable to evidence
* one community suggestion, one reflection, one closing note

In [ ]:
pathway = LearningPathway(
    family_reflection=(
        "Your family is trading a rigid timetable for a rhythm that still keeps "
        "the reading you love, with more room outdoors for animals and drawing."
    ),
    starting_rhythm=[
        RhythmPractice(
            timing="Most mornings",
            practice="Take a short walk before anything else begins.",
            why_it_fits="It adds outdoor time without adding a schedule.",
        ),
        RhythmPractice(
            timing="Once a week",
            practice="Keep a shared drawing journal of what you noticed.",
            why_it_fits="It connects the walk to your child's love of drawing.",
        ),
    ],
    resources=[
        ResourceRecommendation(
            title="Start an interest catalog",
            why_it_fits="It gives the animal interest somewhere to grow.",
            source_id="synthetic-guide-0007",
            url=None,
        ),
        ResourceRecommendation(
            title="Build a gentle weekly rhythm",
            why_it_fits="It replaces the timetable you are leaving behind.",
            source_id="synthetic-guide-0011",
            url=None,
        ),
    ],
    community_suggestion=CommunitySuggestion(
        suggestion="Visit one informal nature meetup this month.",
        why_it_fits="It is low pressure and matches your outdoor goal.",
        source_id="synthetic-guide-0011",
    ),
    closing_note="Go slowly. One walk and one journal page is a real start.",
)

print("rhythm practices:", len(pathway.starting_rhythm))
print("resources       :", len(pathway.resources))
print("cited source ids:", [resource.source_id for resource in pathway.resources])

## Failure path: a pathway that breaks the contract

A single resource is not enough to be useful, so the schema refuses it. When the
language model is asked to produce this schema directly, the same rule becomes a
generation constraint rather than a hope expressed in the prompt.

In [ ]:
try:
    LearningPathway(
        family_reflection=pathway.family_reflection,
        starting_rhythm=pathway.starting_rhythm,
        resources=pathway.resources[:1],
        community_suggestion=pathway.community_suggestion,
        closing_note=pathway.closing_note,
    )
except ValidationError as error:
    for item in error.errors():
        print(list(item["loc"]), "->", item["msg"])

## `RetrievedRecord` and `GroundedPathwayResult`: keeping evidence attached

`GroundedPathwayResult` is the contract that makes the system auditable. It
carries the intake, the exact retrieval query, the evidence that came back, and
the pathway that was produced from it.

Because the evidence travels with the answer, later stages can ask a question
that would otherwise be impossible: *was every cited id actually retrieved?*

In [ ]:
retrieved = [
    RetrievedRecord(record=record, score=0.81),
    RetrievedRecord(
        record=SourceRecord.model_validate(
            record.model_dump()
            | {
                "source_id": "synthetic-guide-0011",
                "title": "Synthetic guide, chunk 11",
            }
        ),
        score=0.74,
    ),
]

grounded = GroundedPathwayResult(
    intake=one_child_intake,
    retrieval_query="Children: older child (age 12) interested in animals, drawing",
    retrieved_records=retrieved,
    pathway=pathway,
)

retrieved_ids = {item.record.source_id for item in grounded.retrieved_records}
cited_ids = {resource.source_id for resource in grounded.pathway.resources}

print("retrieved:", sorted(retrieved_ids))
print("cited    :", sorted(cited_ids))
print("every citation was retrieved:", cited_ids <= retrieved_ids)

## Key takeaways

* The models module is the project's shared vocabulary; every other module
  imports from it rather than inventing its own dictionaries.
* Constraints encode product decisions, not just type safety: age ranges, list
  minimums, and pathway sizes are all deliberate.
* Chunk identity and source identity are different things, and retrieval depends
  on that difference.
* `GroundedPathwayResult` keeps evidence attached to output, which is what makes
  grounding checks and evaluation possible at all.

## Next

Notebook 2 adds the first moving part: generating a pathway with Anthropic Claude
from hand-assembled context, before any retrieval exists.